# Feature Extraction  
This notebook covers all implemented feature extraction techniques for the project. These features include linguistic and contextual embeddings, each derived using different methodologies to prepare the data for downstream analysis.

### Implemented Features:
1. **Sinusoidal Positional Embedding**: Computes positional embeddings based on sinusoidal functions, typically for representing word positions in a sequence.
2. **GloVe Embedding Extraction**: Generates high-dimensional word embeddings based on pre-trained GloVe vectors to capture semantic relations.
3. **Mel Spectrogram Features**: Extracts frequency domain features using Mel spectrogram representations of audio inputs.
4. **GPT-2 XL Contextual Embeddings**: Extracts hidden state representations across multiple layers of GPT-2 XL for word-level context modeling.
5. **Wav2Vec2.0 Word-Level Features**: Processes audio to generate embeddings based on Wav2Vec2.0's learned representations.
6. **Residual Context Features**: Captures contextual information by modeling the relationship between stimulus and response using temporal ridge regression.

Each method follows a pipeline to load the data, process it into the corresponding feature representations, and save the results for downstream tasks.

## 0. Load TIMIT File Names

In [2]:
import sys
sys.path.append("..")
import os

timit_dir = "../data/timit_example"
timit_file = os.path.join(timit_dir, "TIMIT_examples.txt")

# Storage for extracted features
all_word_feats = []
all_sent_feats = []

# Step 1: Load file names from TIMIT_examples.txt
with open(timit_file, "r") as f:
    timit_names = [line.strip() for line in f.readlines()]
print(f"Loaded {len(timit_names)} files from {timit_file}")

Loaded 8 files from ../data/timit_example/TIMIT_examples.txt


In [2]:
timit_names

['mjac0_si2148',
 'mjmm0_si625',
 'msat0_si896',
 'frew0_si1910',
 'fmah1_si2139',
 'mbsb0_si1983',
 'fdml0_si1779',
 'mtrt0_si597']

## 1. Sinusoidal Word Positional Embedding Extraction

In [5]:
from brain_encoding.feature_extraction import extract_sinusoidal_embeddings

output_file = "../data/example_sinusoidal_embeddings.pkl"
d_model = 50

# Extract sinusoidal embeddings
final_embeddings = extract_sinusoidal_embeddings(timit_names, timit_dir, output_file, d_model)
print(f"Final embeddings shape: {final_embeddings.shape}")

Embeddings saved at ../data/example_sinusoidal_embeddings.pkl
Final embeddings shape: (52, 50)


## 2. GloVe Feature Extraction

In [4]:
# Import required modules
import os
import pickle
from brain_encoding.util import load_glove_embeddings, parse_wrd_file
from brain_encoding.feature_extraction import extract_glove_features_from_wrd
from tensorflow.keras.preprocessing.text import Tokenizer


# Step 2: Set up paths
timit_dir = "../data/timit_example/"  # Path to TIMIT .wrd files
glove_path = "../data/glove.6B.300d.txt"  # Path to GloVe embeddings
embedding_dim = 300  # Dimension of GloVe embeddings
output_file = "../data/example_glove_wemb300d.pkl"  # Output file for GloVe features

# Step 3: Build `token_set` from TIMIT `.wrd` files using parse_wrd_file
token_set = set()
for i, file_name in enumerate(timit_names):
    print(f"Processing file {i + 1}/{len(timit_names)}: {file_name}")
    wrd_file_path = os.path.join(timit_dir, file_name + ".wrd")
    try:
        words, _ = parse_wrd_file(wrd_file_path)  # Use parse_wrd_file to extract words
        token_set.update(words)  # Add words to the token set
    except FileNotFoundError:
        print(f"Warning: {wrd_file_path} not found, skipping.")

print(f"Total unique words in token set: {len(token_set)}")

# Step 4: Create a tokenizer and word_index
tokenizer = Tokenizer()
tokenizer.fit_on_texts(token_set)  # Fit tokenizer on the unique set of words
word_index = tokenizer.word_index  # Word-to-index mapping
print(f"Vocabulary size: {len(word_index)}")

# Step 5: Load GloVe embeddings
embedding_matrix = load_glove_embeddings(glove_path, word_index, embedding_dim)
print(f"GloVe embeddings loaded with shape: {embedding_matrix.shape}")

# Step 6: Extract GloVe features using the feature extraction module
glove_features = extract_glove_features_from_wrd([timit_names], timit_dir, word_index, embedding_matrix)
print(f"GloVe feature matrix shape: {glove_features['feat'].shape}")
print(f"Number of word identifiers: {len(glove_features['word_seq_list'])}")

# Step 7: Save the results to a file
with open(output_file, "wb") as f:
    pickle.dump(glove_features, f)
print(f"GloVe features successfully saved to: {output_file}")

# Step 8: Visualize some sample results
# Show a few word embeddings and corresponding identifiers
print("\nSample word embeddings:")
for i in range(min(5, len(glove_features['word_seq_list']))):
    print(f"{glove_features['word_seq_list'][i]}: {glove_features['feat'][i][:10]} ...")  # Print first 10 dimensions

2026-05-06 15:42:00.344521: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-06 15:42:02.292990: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 15:42:02.980754: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-05-06 15:42:02.980811: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore 

Processing file 1/8: mjac0_si2148
Processing file 2/8: mjmm0_si625
Processing file 3/8: msat0_si896
Processing file 4/8: frew0_si1910
Processing file 5/8: fmah1_si2139
Processing file 6/8: mbsb0_si1983
Processing file 7/8: fdml0_si1779
Processing file 8/8: mtrt0_si597
Total unique words in token set: 43
Vocabulary size: 43
GloVe embeddings loaded with shape: (44, 300)
Processing block 1/1...
  Processing file: mjac0_si2148
  Processing file: mjmm0_si625
  Processing file: msat0_si896
  Processing file: frew0_si1910
  Processing file: fmah1_si2139
  Processing file: mbsb0_si1983
  Processing file: fdml0_si1779
  Processing file: mtrt0_si597
GloVe feature matrix shape: (52, 300)
Number of word identifiers: 52
GloVe features successfully saved to: ../data/glove_wemb300d_all_timit.pkl

Sample word embeddings:
look_0: [-0.11653    -0.02557    -0.081566   -0.22752     0.20206     0.25826001
 -0.30046001 -0.1432      0.2498     -1.7184    ] ...
somewhere_0: [-0.82871997  0.13518    -0.4656899

## 3. Mel Feature Extraction

In [2]:
from brain_encoding.feature_extraction import extract_word_level_mel_features

# Define parameters
output_file = "../data/example_word_mel_features.pkl"  # Path to save word-level Mel features
n_fft = 400         # FFT window size
hop_length = 160    # Hop size (10ms for 160 samples at 16kHz)
n_mels = 128        # Number of Mel bands

# Extract word-level Mel features
mel_word_features = extract_word_level_mel_features(timit_names, timit_dir, output_file, n_fft, hop_length, n_mels)

# Print shape of the final feature matrix
print(f"Final word-level Mel features shape: {mel_word_features.shape}")

Embeddings saved at ../data/example_word_mel_features.pkl
Final word-level Mel features shape: (52, 128)


## 4. GPT2XL Feature Extraction

In [5]:
import torch
from transformers import GPT2Tokenizer, GPT2Model
import os
import numpy as np
from brain_encoding.util import get_merged_hidden_states
import pickle 

# Define the mapping of layer indices to layer names
layer_name_dict = {
    0: "fs_ext", 1: "decoder0", 5: "decoder4", 9: "decoder8",
    13: "decoder12", 17: "decoder16", 21: "decoder20",
    25: "decoder24", 29: "decoder28", 33: "decoder32",
    37: "decoder36", 41: "decoder40", 45: "decoder44", 48: "decoder47"
}

cache_dir = "../cache"
os.makedirs(cache_dir, exist_ok=True)

# Initialize GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-xl', cache_dir=cache_dir)
model = GPT2Model.from_pretrained('gpt2-xl', cache_dir=cache_dir)
model.eval()  # Ensure the model is in evaluation mode

# Input directory containing TIMIT `.wrd` files
timit_dir = "../data/timit_example"

# Dictionary to store extracted features for each saved layer
feat_dict = {layer_name: [] for layer_name in layer_name_dict.values()}  # Initialize with layer names

# Loop over all files in `timit_names`
for file_name in timit_names:
    # Step 1: Read tokens from the TIMIT `.wrd` file
    wrd_file_path = os.path.join(timit_dir, file_name + ".wrd")
    with open(wrd_file_path, "r") as f:
        lines = f.readlines()
    true_tokens = [line.strip().split()[-1] for line in lines]  # Extract true tokens
    sentence = " ".join(true_tokens)  # Reconstruct the sentence

    # Step 2: Tokenize the sentence and pass it through the model
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():  # Disable gradients (inference mode)
        outputs = model(**inputs, output_hidden_states=True)

    hidden_states = outputs.hidden_states  # Hidden states of all layers
    tokens_from_id = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])  # GPT-2 tokenized output

    # Step 3: Merge subword hidden states into word-level hidden states
    merged_hidden_states = get_merged_hidden_states(tokens_from_id, true_tokens, hidden_states)

    # Step 4: Extract features for all layers specified in `layer_name_dict`
    for layer_idx, layer_name in layer_name_dict.items():
        feat_dict[layer_name].append(merged_hidden_states[layer_idx])  # Save merged features for this layer

# Step 5: Combine features for each layer into a single matrix
final_feat_dict = {}
for layer_name, features in feat_dict.items():
    final_feat_dict[layer_name] = torch.from_numpy(np.concatenate(features, axis=0))  # Combine into a matrix

# Feature dictionary for all specified layers
for layer_name, feature_matrix in final_feat_dict.items():
    print(f"{layer_name}: {feature_matrix.shape}")
    
# Step 6: Save the results to a file
output_file = "../data/example_gpt2xl_features.pkl"

with open(output_file, "wb") as f:
    pickle.dump(final_feat_dict, f)
print(f"Features successfully saved to: {output_file}")

fs_ext: torch.Size([52, 1600])
decoder0: torch.Size([52, 1600])
decoder4: torch.Size([52, 1600])
decoder8: torch.Size([52, 1600])
decoder12: torch.Size([52, 1600])
decoder16: torch.Size([52, 1600])
decoder20: torch.Size([52, 1600])
decoder24: torch.Size([52, 1600])
decoder28: torch.Size([52, 1600])
decoder32: torch.Size([52, 1600])
decoder36: torch.Size([52, 1600])
decoder40: torch.Size([52, 1600])
decoder44: torch.Size([52, 1600])
decoder47: torch.Size([52, 1600])
Features successfully saved to: ../data/example_gpt2xl_features.pkl


## 5. Wav2Vec2.0 Feature Extraction

In [4]:
import pickle
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from brain_encoding.feature_extraction import extract_wav2vec2_features
from brain_encoding.util import parse_wrd_file, compute_compression_ratio, segment_features
import numpy as np 

# Define paths and parameters
model_checkpoint = "facebook/wav2vec2-base"
audio_dir = "../data/timit_example"
output_path = "../data/example_wav2vec2_word_feats.pkl"
sample_rate = 16000  # Expected audio sample rate

cache_dir = "../cache"
os.makedirs(cache_dir, exist_ok=True)

# Load Wav2Vec2 model
print("Loading Wav2Vec2 model...")
processor = Wav2Vec2Processor.from_pretrained(model_checkpoint, cache_dir=cache_dir)
model = Wav2Vec2ForCTC.from_pretrained(model_checkpoint, cache_dir=cache_dir)

# Define input data
wav2vec2_features = ["fs_ext", "fs_proj"] + [f"encoder{i}" for i in range(13)]
word_embeddings = {feat: [] for feat in wav2vec2_features}

# Process each audio file
for i, filename in enumerate(timit_names):
    print(f"Processing file {i + 1}/{len(timit_names)}: {filename}")
    
    # Load audio and word alignment files
    wav_path = f"{audio_dir}/{filename}.wav"
    wrd_path = f"{audio_dir}/{filename}.wrd"
    speech_array, sampling_rate = torchaudio.load(wav_path)
    
    if sampling_rate != sample_rate:
        speech_array = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=sample_rate)(speech_array)
    
    cur_tokens, start_end_list = parse_wrd_file(wrd_path)
    
    # Extract features
    features = extract_wav2vec2_features(model=model, speech_array=speech_array, sample_rate=sample_rate)
    
    # Compute compression ratio
    compressed_len = features["proj"].shape[0]
    compression_ratio = compute_compression_ratio(original_len=speech_array.shape[2], compressed_len=compressed_len)
    
    # Segment features by word-level timestamps
    segmented = segment_features(features=features, start_end_list=start_end_list, compression_ratio=compression_ratio)
    
    # Collect embeddings for each word and feature type
    for token_i, token in enumerate(cur_tokens):
        word_embeddings["fs_ext"].append(segmented["ext"][token_i])
        word_embeddings["fs_proj"].append(segmented["proj"][token_i])
        for layer_idx, layer_embedding in enumerate(segmented["encoder"]):
            word_embeddings[f"encoder{layer_idx}"].append(layer_embedding[token_i])

# Convert lists to matrices for each feature type
for feat in wav2vec2_features:
    word_embeddings[feat] = np.array(word_embeddings[feat])  # Shape: num_words x vector_dim

# Save word embeddings
print(f"Saving word embeddings to {output_path}...")
with open(output_path, "wb") as f:
    pickle.dump(word_embeddings, f)

print("Feature extraction completed!")

Loading Wav2Vec2 model...


/root/anaconda3/envs/py37/lib/python3.7/site-packages/transformers/configuration_utils.py:370: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  "Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 "
Some weights of the model checkpoint at facebook/wav2vec2-base were not used when initializing Wav2Vec2ForCTC: ['project_q.weight', 'project_hid.weight', 'quantizer.weight_proj.bias', 'quantizer.weight_proj.weight', 'quantizer.codevectors', 'project_q.bias', 'project_hid.bias']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- T

Processing file 1/8: mjac0_si2148
Processing file 2/8: mjmm0_si625
Processing file 3/8: msat0_si896
Processing file 4/8: frew0_si1910
Processing file 5/8: fmah1_si2139
Processing file 6/8: mbsb0_si1983
Processing file 7/8: fdml0_si1779
Processing file 8/8: mtrt0_si597
Saving word embeddings to ../data/example_wav2vec2_word_feats.pkl...
Feature extraction completed!


## 6. Residual context Feature Extraction

In [1]:
import sys
sys.path.append("..")
import pickle
import numpy as np
from brain_encoding.temporal_receptive_field import get_delays, get_dstim, get_alphas, run_cv_word_lag_temporal_ridge_regression_model

# Generate a random feature matrix as a placeholder
# In practice, replace this matrix with GPT-2 XL feature matrix
# (In this study, the stimulus uses `fs_ext` (i.e., the first layer of GPT-2 embeddings),
# and the response uses `decoder8` (a middle layer of GPT-2 decoder embeddings)). 
num_words = 3575  # Number of words (rows)
vector_dim = 1600  # Feature vector dimension (columns)
example_fs_ext_feat = np.random.rand(num_words, vector_dim)

# Exclude t=0 to predict the response feature at t-1
stim = example_fs_ext_feat[1:, :]  
print(f"Stim feature matrix shape (placeholder): {stim.shape}")

# Example settings for temporal delay and data sampling rates
ds, fs = 0.25, 100
delays = get_delays(delay_seconds=ds, fs=fs)

# Preprocess the feature matrix to obtain delayed stimulus representation
# Concatenate word embeddings from time steps t-24 to t
# The delayed stimulus (`dstim`) will have dimensions (num_words-1, vector_dim * 25)
add_edges = False
dstim = get_dstim(stim, delays, add_edges=add_edges)
print(f"Delayed stimulus shape: {dstim.shape}")

# Generate a random response matrix as a placeholder
# Real response matrix should match the number of time points in `delayed stimulus` (after accounting for delays)
example_decoder8_feat = np.random.rand(num_words, vector_dim)

# Response matrix: Exclude the last time step to align with the stimulus
resp = example_decoder8_feat[:-1, :]  # Response matrix (rows: time points, cols: channels)
print(f"Response matrix shape (placeholder): {resp.shape}")

# Run cross-validated word-lag temporal ridge regression
test_corr_folds, wts_folds, best_alphas, pred_all, bs_folds = run_cv_word_lag_temporal_ridge_regression_model(
    dstim,         # Delayed stimulus matrix
    resp,          # Response matrix
    alphas=get_alphas(-5, 5, 10),  # Ridge regression alpha values
    n_folds=5,                   # Number of folds for cross-validation
    apply_pca=True,              # Apply PCA to reduce dimensions
    variance_ratio=0.95,         # Retain 95% variance during PCA
    scale=True                   # Z-score normalization
)

# Display the cross-validation performance
print("Finished running cross-validated temporal ridge regression.")

# Compute residual contextual feature
# Subtract predicted responses from observed responses to obtain residuals
res_context = resp - pred_all  

# Save residual contextual features to file
output_path = "../data/example_residual_context_features.pkl"
with open(output_path, "wb") as f:
    pickle.dump(res_context, f)
print(f"Residual contextual features saved to {output_path}")


Stim feature matrix shape (placeholder): (3574, 1600)
Delayed stimulus shape: (3574, 40000)
Response matrix shape (placeholder): (3574, 1600)
Running fold 0. 
 pca train shape: (2287, 40000), test shape: (715, 40000), valid shape: (572, 40000)
dstim PCA: 0.95 components: 2106
scale_and_pca consume time: 116.20620679855347 s
train shape: (2287, 2106), test shape: (715, 2106), val_shape: (572, 2106)
best_wts_mat shape: (2106, 1600)
best_bs_mat shape: (1600,)

Running fold 1. 
 pca train shape: (2287, 40000), test shape: (715, 40000), valid shape: (572, 40000)
dstim PCA: 0.95 components: 2099
scale_and_pca consume time: 115.88398003578186 s
train shape: (2287, 2099), test shape: (715, 2099), val_shape: (572, 2099)
best_wts_mat shape: (2099, 1600)
best_bs_mat shape: (1600,)

Running fold 2. 
 pca train shape: (2287, 40000), test shape: (715, 40000), valid shape: (572, 40000)
dstim PCA: 0.95 components: 2099
scale_and_pca consume time: 113.3459460735321 s
train shape: (2287, 2099), test sha